In [ ]:
# [화면 설정 전용 (양자 연산과 무관)] 먼저 이 셀을 실행(Shift+Enter)하면 아래 마크다운 글자가 1.35배 확대됨
from __future__ import annotations
from IPython.display import HTML, display

display(HTML('''
<style>
.jp-RenderedMarkdown { font-size: 1.35em !important; line-height: 1.65 !important; }
.jp-RenderedMarkdown code { font-size: 0.90em !important; }
</style>
'''))

# [양자컴퓨팅 부트캠프] Day 3 (2부): Qiskit으로 실행하는 다중 큐비트 Grover 탐색

1부에서 다룬 Grover 탐색 알고리즘을 8큐비트($N=256$)와 16큐비트($2^{16} = 65,536$차원) 규모로 실행해 동작을 확인하는 실습 노트북임.

### 실습 방식 및 Qiskit 명령어 학습 목적
- **실행·관찰 중심 진행**: AI가 파이썬 스크립트를 대신 작성해 주는 환경이므로, 본 노트북은 직접 타이핑 대신 완성된 셀을 순서대로 실행(`Shift+Enter`)하며 원리와 결과를 해석.
- **Qiskit 명령어 독해 필요성**: AI가 만든 코드에서 비트 순서나 게이트 적용 대상이 이론과 어긋날 때 직접 읽고 검산해야 하며, AI 사용이 어려운 상황에서도 필요한 블록을 복사해 게이트와 수치를 수정할 수 있어야 함.
- **NumPy 대신 Qiskit 사용 이유**: 16큐비트 기준 NumPy 게이트 행렬($2^{16} \times 2^{16}$)은 1개당 약 68.7 GB 메모리가 필요해 일반 PC에서 생성 불가. Qiskit은 길이 $2^{16}$ 상태벡터(약 1.0 MB)에 게이트 규칙을 순차 적용해 수 초 내 계산 완료.
- **코드 주석 구분 안내**: 각 코드 셀은 핵심 양자 연산(`# [Qiskit ...]`)과 그래프·표 출력 코드(`# [출력·시각화 전용 (양자 연산과 무관)]`)를 주석으로 분리 표기함.

### 2부 진행 구성
1. **[안내] Part 0. Qiskit 핵심 명령어 입문**: `QuantumCircuit`, `draw`, `Statevector`, Little-Endian 비트 순서, $\mathrm{MCZ}$·Diffusion 조립
2. **[관찰 1] Toy Example Grover ($n=8, N=256, M=1$)**: $k^*=12$ 최종 분포 및 12회 `Oracle -> Diffusion` 단계별 진폭·확률 추적
3. **[관찰 2] 반복 횟수 $k$에 따른 증폭과 과회전(Over-rotation) 곡선**: `sv.evolve()`로 $k=0\sim 30$ 추적 및 이론 사인파 대조
4. **[관찰 3] 검산 규칙 기반 Grover: 8-노드 2-색칠 (16큐비트, $M=2$)**: 오라클 회로도 독해, 두 해(`01010101`, `10101010`) 검출 및 임시 보조 큐비트 정리(Uncomputation) 비교
5. **[관찰 4] Shot Noise 통계적 수렴 비교**: 64 / 512 / 2,048 Shots 표본 분포 비교

In [ ]:
# [환경 설정] Qiskit 및 수치·시각화 패키지 불러오기
from __future__ import annotations
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZGate
from qiskit.quantum_info import Statevector

# [메모리 계산 (양자 연산과 무관, 비교용)] 16큐비트 complex128(16 bytes), SI 10진 GB/MB 기준
n_comp = 16
dim_comp = 2 ** n_comp
numpy_matrix_gb = (dim_comp * dim_comp * 16) / (1000 ** 3)
qiskit_sv_mb = (dim_comp * 16) / (1000 ** 2)

# [출력·시각화 전용 (양자 연산과 무관)] 버전 및 메모리 비교표 출력
display(Markdown(
    f"**Qiskit 실습 환경 준비 완료**\n\n"
    f"- `qiskit` 버전: `{qiskit.__version__}` / `numpy` 버전: `{np.__version__}`\n\n"
    f"| 비교 항목 ($n=16$ 큐비트, $2^{{16}} = {dim_comp:,}$차원) | 저장 대상 | 필요 메모리 | 비고 |\n"
    f"| :--- | :--- | :---: | :--- |\n"
    f"| **NumPy 행렬 방식** | $2^{{16}}\\times 2^{{16}}$ 게이트 행렬 1개 | **`{numpy_matrix_gb:.1f} GB`** | 일반 노트북 실행 불가 |\n"
    f"| **Qiskit 상태벡터 방식** | 길이 $2^{{16}}$ 상태벡터 | **`{qiskit_sv_mb:.1f} MB`** | 노트북에서 수 초 내 실행 |"
))

## [안내 0-1] Qiskit 필수 명령어 치트시트 & 2일차 Bell 상태 다시 만들기

- **핵심 명령어 8종 요약** (코드 독해·부분 복붙용)

| 명령어 | 1·2일차 개념 대응 | 역할 |
| :--- | :--- | :--- |
| `qc = QuantumCircuit(n)` | $n$큐비트 초기 상태 $\ket{0}^{\otimes n}$ | $n$개 큐비트 선(wire)을 가진 빈 양자 회로 생성 |
| `qc.h(q)`, `qc.x(q)`, `qc.z(q)` | 단일 큐비트 게이트 $H, X, Z$ | 지정 큐비트 `q`(또는 `range(n)`)에 1큐비트 게이트 추가 |
| `qc.cx(ctrl, tgt)` | 2큐비트 $\mathrm{CX}$ (CNOT) | `ctrl`이 $\ket{1}$일 때 `tgt`에 $X$ 적용 |
| `qc.mcx(ctrls, tgt)` / `ZGate().control(k)` | 다중 제어 $\mathrm{MCX}$, $\mathrm{MCZ}$ | 제어 큐비트들이 모두 $\ket{1}$일 때 타깃에 $X$ 또는 $Z$ 적용 |
| `qc.draw("mpl")` | 양자 회로도 시각화 | 왼쪽$\to$오른쪽 시간 순서로 게이트 배치 출력 |
| `Statevector.from_instruction(qc)` / `sv.evolve(step)` | 상태벡터 계산 및 순차 연산 | 최종 상태벡터 계산(`evolve`는 현재 상태에 1단계 추가 적용) |
| `sv.data` / `sv.probabilities()` / `sv.probabilities_dict()` | **[시뮬레이터 전용 장점]** 복소 진폭·이론 확률 직접 조회 | 실제 양자 칩에서는 볼 수 없는 정확한 진폭(`sv.data`)과 이론 확률 전체를 즉시 조회(검증용) |
| `sv.sample_counts(S)` | **[실제 칩 모사]** Born 규칙 유한 측정 ($S$ Shots) | 실제 양자 컴퓨터처럼 $S$번 측정해 얻은 관측 빈도 딕셔너리 반환 |

- **실습 내용**: 2일차에 NumPy로 만든 Bell 상태 $\ket{\Phi^+} = \frac{\ket{00}+\ket{11}}{\sqrt{2}}$를 `qc.h(0)`, `qc.cx(0, 1)`로 구성. 시뮬레이터 전용 장점(`probabilities_dict()`)으로 정확한 50:50 확률을 먼저 보고, 실제 양자 칩과 같은 유한 측정(`sample_counts(1000)`)과 비교.
- **출력 구성**:
  1. 회로 그림: 위쪽 `q_0`, 아래쪽 `q_1`에 `H`와 CNOT(`•`, `⊕`)이 왼쪽에서 오른쪽 순으로 배치됨.
  2. 요약 표: 이론 확률(`'00'` 50%, `'11'` 50%)과 `sample_counts(1000)` 표본 빈도 대조.

In [ ]:
# [Qiskit 회로 구성] 2큐비트 Bell 상태 |Phi+> 생성
qc_bell = QuantumCircuit(2)
qc_bell.h(0)        # q_0에 Hadamard 적용
qc_bell.cx(0, 1)    # q_0(제어) -> q_1(타깃) CNOT 적용

# [Qiskit 회로도 시각화]
fig_bell = qc_bell.draw("mpl")

# [Qiskit 상태벡터·측정] 이론 확률 및 1,000 Shots 샘플링
sv_bell = Statevector.from_instruction(qc_bell)
probs_bell = sv_bell.probabilities_dict()
sv_bell.seed(42)
counts_bell = sv_bell.sample_counts(1000)

# [자가 검증]
assert np.isclose(probs_bell["00"], 0.5) and np.isclose(probs_bell["11"], 0.5), "Bell 상태 확률 오류"

# [출력·시각화 전용 (양자 연산과 무관)] 회로 그림 및 비교 표 화면 출력
display(fig_bell)
display(Markdown(
    "**[안내 0-1 결과] Qiskit으로 만든 2일차 Bell 상태 $\\ket{\\Phi^+}$**\n\n"
    "| 비트열 | 이론 확률 (`probabilities_dict`) | 1,000 Shots 관측 (`sample_counts`) |\n"
    "| :---: | :---: | :---: |\n"
    f"| `00` | `{probs_bell.get('00', 0.0)*100:.1f}%` | `{counts_bell.get('00', 0)}` 회 |\n"
    f"| `01` | `{probs_bell.get('01', 0.0)*100:.1f}%` | `{counts_bell.get('01', 0)}` 회 |\n"
    f"| `10` | `{probs_bell.get('10', 0.0)*100:.1f}%` | `{counts_bell.get('10', 0)}` 회 |\n"
    f"| `11` | `{probs_bell.get('11', 0.0)*100:.1f}%` | `{counts_bell.get('11', 0)}` 회 |"
))

## [안내 0-2] Qiskit 비트 순서 규칙: Little-Endian

- **표기 차이**: 1·2일차와 1부 슬라이드는 왼쪽부터 읽는 Big-Endian($\ket{q_0 q_1}$) 사용. Qiskit은 오른쪽이 `q_0`, 왼쪽이 마지막 큐비트인 Little-Endian($\ket{q_{n-1}\cdots q_1 q_0}$) 사용.
- **실습 내용**: 2큐비트 회로 `qc_endian`에서 `q_0`에만 `qc.x(0)`을 적용해 `1`로 반전한 뒤 측정 결과 확인.
- **출력 구성**:
  1. 회로 그림: `q_0`에만 `X` 게이트가 놓인 모습.
  2. 요약 결과: `q_0 q_1` 기준 `'10'`이 아니라, 왼쪽 `q_1=0`·오른쪽 `q_0=1` 순서인 `'01'`이 100% 관측됨(뒤의 Toy Grover에서 `reversed(target)`을 쓰는 이유).

In [ ]:
# [Qiskit 회로 구성] q_0만 |0> -> |1>로 반전 (q_1은 |0> 유지)
qc_endian = QuantumCircuit(2)
qc_endian.x(0)

# [Qiskit 회로도 시각화]
fig_end = qc_endian.draw("mpl")

# [Qiskit 상태벡터·검증] 출력 비트열이 '01'인지 확인
sv_endian = Statevector.from_instruction(qc_endian)
probs_e = sv_endian.probabilities_dict()
assert np.isclose(probs_e.get("01", 0.0), 1.0), "Little-Endian 확인 실패"

# [출력·시각화 전용 (양자 연산과 무관)]
display(fig_end)
display(Markdown(
    "**[안내 0-2 결과] `qc.x(0)` 실행 후 Qiskit 출력 비트열 확인**\n\n"
    f"- 관측된 상태 및 확률: `{ {str(k): round(float(v), 4) for k, v in probs_e.items()} }`\n"
    "- **읽는 법**: Qiskit 문자열 `'01'` $\\implies$ 왼쪽 `q_1 = 0`, 오른쪽 `q_0 = 1` (**Little-Endian**)"
))

## [안내 0-3] 다중 제어 게이트($\mathrm{MCZ}$)와 Diffusion 조립 (3큐비트 미니 검증)

- **핵심 구현 요소** (1부 슬라이드 22~24 대응):
  - `apply_mcz`: `qc.append(ZGate().control(len(qubits) - 1), qubits)`로 모든 큐비트가 `1`일 때만 위상을 $-1$ 반전.
  - `apply_diffusion`: $H^{\otimes n} X^{\otimes n} \mathrm{MCZ} X^{\otimes n} H^{\otimes n}$ 적용(측정 확률과 무관한 전역 부호 $-1$ 생략).
  - `qc.barrier()`: 초기 중첩, Oracle, Diffusion 구역을 나누는 시각적 구분선(실제 연산 미변화).
- **실습 내용**: 3큐비트($N=8$, 정답 `'101'`) 회로에 Grover 1회 적용.
- **출력 구성**:
  1. 3큐비트 회로도: `H` $\to$ `barrier` $\to$ `X`-샌드위치 `MCZ` 오라클 $\to$ Diffusion 순서 확인.
  2. 확률 비교 표: 초기 $12.5\%$($1/8$)이던 `'101'` 확률이 1회 만에 이론값 $78.125\%$($25/32$)로 증가함 확인.

In [ ]:
# [Qiskit 공용 연산자 함수 정의] 8큐비트·16큐비트 실습에서도 재사용
def apply_mcz(qc: QuantumCircuit, qubits: list[int]) -> None:
    '''지정된 큐비트 목록이 모두 |1>일 때만 위상을 -1 반전 (MCZ)'''
    qc.append(ZGate().control(len(qubits) - 1), qubits)


def apply_diffusion(qc: QuantumCircuit, main_qubits: list[int]) -> None:
    '''n큐비트 Diffusion 연산자 (전역 부호 -1 생략): H^n -> X^n -> MCZ -> X^n -> H^n'''
    qc.h(main_qubits)
    qc.x(main_qubits)
    apply_mcz(qc, main_qubits)
    qc.x(main_qubits)
    qc.h(main_qubits)


def apply_toy_oracle(qc: QuantumCircuit, target_bits: str) -> None:
    '''정답 비트열 target_bits만 -1 반전하는 X-샌드위치 오라클 (Little-Endian)'''
    n_q = len(target_bits)
    for i, bit in enumerate(reversed(target_bits)):
        if bit == "0":
            qc.x(i)
    apply_mcz(qc, list(range(n_q)))
    for i, bit in enumerate(reversed(target_bits)):
        if bit == "0":
            qc.x(i)


# [Qiskit 회로 구성: 3큐비트 미니 검증 (N=8, 정답 '101', 1회 반복)]
qc_mini_init = QuantumCircuit(3)
qc_mini_init.h(range(3))

qc_mini = QuantumCircuit(3)
qc_mini.h(range(3))
qc_mini.barrier()
apply_toy_oracle(qc_mini, "101")
qc_mini.barrier()
apply_diffusion(qc_mini, [0, 1, 2])

# [Qiskit 회로도 시각화]
fig_mini = qc_mini.draw("mpl", scale=0.85)

# [Qiskit 상태벡터 계산 및 자가 검증]
p0 = Statevector.from_instruction(qc_mini_init).probabilities_dict()
p1 = Statevector.from_instruction(qc_mini).probabilities_dict()
assert np.isclose(p1["101"], 25.0 / 32.0), "3큐비트 1회 반복 정답 확률은 25/32여야 함"

# [출력·시각화 전용 (양자 연산과 무관)]
display(fig_mini)
display(Markdown(
    "**[안내 0-3 결과] 3큐비트 미니 Grover ($N=8$, 정답 `'101'`) 1회 적용 검증**\n\n"
    "| 상태 | 초기 중첩 ($k=0$) | 1회 적용 후 ($k=1$) | 비고 |\n"
    "| :---: | :---: | :---: | :--- |\n"
    f"| **`101` (정답)** | `{p0['101']*100:.3f}%` ($1/8$) | **`{p1['101']*100:.3f}%`** ($25/32$) | 1회 적용으로 6.25배 증폭 |\n"
    f"| 나머지 7개 오답 (각각) | `{p0['000']*100:.3f}%` ($1/8$) | `{p1['000']*100:.3f}%` ($1/32$) | 동일하게 억제됨 |"
))

## [관찰 1-1] Toy Example Grover ($n=8, N=256, M=1$): 초기 vs 최적 반복

- **확장 설정**: 3큐비트 검증 함수(`apply_toy_oracle`, `apply_diffusion`)를 동일 구조로 8큐비트($N=256$, 정답 `'10110101'`)에 적용.
- **이론 값**: $\sin\theta = \frac{1}{\sqrt{256}} = \frac{1}{16}$ ($\theta \approx 3.583^\circ$), 최적 반복 횟수 $k^* = \left\lfloor \frac{\pi}{4}\sqrt{256} \right\rfloor = 12$회(고전 평균 128회).
- **출력 구성**:
  1. 상위 Top-5 표 및 2,048 Shots 결과: $k=0$과 $k^*=12$ 확률 비교 및 정답 `'10110101'`(인덱스 `181`) 관측 빈도.
  2. 좌/우 비교 막대 그래프 (가로축 상태 인덱스 `0~255`, 세로축 확률 %): 왼쪽($k=0$)은 `0.39%` 균등 분포, 오른쪽($k^*=12$)은 인덱스 `181`만 약 `99.995%`로 집중됨.

In [ ]:
# [Qiskit Toy Grover 회로 빌더]
n = 8
N = 2 ** n
target = "10110101"
target_idx = int(target, 2)
k_opt = int(np.floor((np.pi / 4.0) * np.sqrt(N)))


def build_toy_grover(n_qubits: int, target_bits: str, iterations: int) -> QuantumCircuit:
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    for _ in range(iterations):
        apply_toy_oracle(qc, target_bits)
        apply_diffusion(qc, list(range(n_qubits)))
    return qc


# [Qiskit 상태벡터·측정 및 자가 검증] k=0 vs k*=12 실행 및 2,048 Shots 샘플링
t0 = time.perf_counter()
sv_init = Statevector.from_instruction(build_toy_grover(n, target, 0))
sv_opt = Statevector.from_instruction(build_toy_grover(n, target, k_opt))
probs_init = sv_init.probabilities()
probs_opt = sv_opt.probabilities()
elapsed = time.perf_counter() - t0

shots = 2048
sv_opt.seed(42)
counts_opt = sv_opt.sample_counts(shots)
target_hits = int(counts_opt.get(target, 0))

assert probs_opt[target_idx] > 0.999, "최적 반복(k=12) 정답 확률은 99.9% 이상이어야 함"

# [출력·시각화 전용 (양자 연산과 무관)] Top-5 확률 표 생성 및 좌/우 막대 그래프 그리기
top5 = np.argsort(probs_opt)[::-1][:5]
rows = "\n".join(
    f"| `{idx:08b}` ({idx}) | {probs_init[idx]*100:.3f}% | **{probs_opt[idx]*100:.4f}%** |"
    for idx in top5
)
display(Markdown(
    f"**[관찰 1-1 결과] Toy Grover ($n=8, N=256, k^*={k_opt}$회, 실행 시간 `{elapsed:.3f}s`)**\n\n"
    f"- 이론 정답 확률 $P(\\texttt{{{target}}})$: **`{probs_opt[target_idx]*100:.4f}%`**\n"
    f"- Qiskit `sample_counts({shots})` 결과: **{target_hits} / {shots} 회** (`{target_hits/shots*100:.2f}%`)\n\n"
    f"| 비트열 (10진수 인덱스) | 초기 확률 ($k=0$) | 최적 반복 후 ($k={k_opt}$) |\n"
    f"| :---: | :---: | :---: |\n"
    f"{rows}"
))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8), sharey=True)
axes[0].bar(range(N), probs_init * 100, width=1.0, color="#6c757d")
axes[0].set_title(r"Initial Superposition ($k=0$, Uniform $1/256 \approx 0.39\%$)")
axes[0].set_xlabel("State Index (0 ~ 255)")
axes[0].set_ylabel("Probability (%)")

axes[1].bar(range(N), probs_opt * 100, width=1.0, color="#0072ce")
axes[1].annotate(
    f"Target {target}\n(idx={target_idx}: {probs_opt[target_idx]*100:.3f}%)",
    xy=(target_idx, probs_opt[target_idx] * 100),
    xytext=(target_idx - 105, 84),
    arrowprops={"arrowstyle": "->", "color": "#1a365d", "lw": 1.2},
    fontsize=9,
    fontweight="bold",
    color="#1a365d",
)
axes[1].set_title(rf"After Grover $k^*={k_opt}$ Iterations")
axes[1].set_xlabel("State Index (0 ~ 255)")
axes[1].set_ylim(0, 108)
plt.tight_layout()
plt.show()

## [관찰 1-2] 12회 반복 전체 추적: `Oracle -> Diffusion` 단계별 진폭·확률 변화

- **관찰 목적**: $k^*=12$회 동안 매 반단계(`Init -> O1 -> D1 -> O2 -> D2 -> ... -> O12 -> D12`, 총 25개 시점)마다 정답 진폭 $a_\omega$, 오답 진폭 $a_{\text{other}}$, 정답 확률 $P(\omega)$가 바뀌는 과정을 직접 추적.
- **사용 명령 (시뮬레이터 전용 장점)**:
  - `qc_ora`, `qc_diff`를 1회분 회로로 분리한 뒤 `sv = sv.evolve(...)`로 교대 적용.
  - 실제 양자 칩에서는 볼 수 없는 시뮬레이터 데이터 `sv.data`(복소 진폭 부호 포함)와 `sv.probabilities()`를 매 반단계마다 즉시 기록.
  - 1부 슬라이드 23의 전역 부호 $(-1)$ 생략 때문에 `apply_diffusion` 통과마다 상태 전체 부호가 $-1$씩 교대하므로, 이론식($2\bar{a}-a_x$)과 부호를 맞추기 위해 출력 시 $(-1)^k$만 보정(확률 $|\cdot|^2$은 동일).
- **출력 구성**:
  1. 13행 추적 표 (`Init` + `k=1..12`): Oracle 직후 부호만 음수로 반전(확률 불변)되고 Diffusion 직후 확률이 계단처럼 뛰어오르는 수치 확인.
  2. 2단 추적 그래프 (가로축 25개 반단계 `Init ~ D12`):
     - 위쪽(정답 진폭 $a_\omega$): Oracle마다 `-`로 뒤집히고 Diffusion마다 `+`로 커지는 톱니파 및 0으로 수렴하는 오답 진폭(회색 점선) 확인.
     - 아래쪽(정답 확률 %): Oracle마다 평평하게 유지되고 Diffusion마다 계단식으로 `0.39% -> 3.48% -> ... -> 99.99%`까지 상승하는 과정 확인.

In [ ]:
# [Qiskit 반단계 회로 구성] 초기(|s>), Oracle(O), Diffusion(D) 각각 분리
qc_step_init = QuantumCircuit(n)
qc_step_init.h(range(n))

qc_ora = QuantumCircuit(n)
apply_toy_oracle(qc_ora, target)

qc_diff = QuantumCircuit(n)
apply_diffusion(qc_diff, list(range(n)))

# [Qiskit 순차 진화 & 시뮬레이터 진폭·확률 추적] Init -> O1 -> D1 -> ... -> O12 -> D12
other_idx = 0   # 오답 대표 인덱스 (00000000 != 10110101)
sv_step = Statevector.from_instruction(qc_step_init)

labels = ["Init"]
is_oracle = [False]
amp_w = [float(sv_step.data[target_idx].real)]
amp_other = [float(sv_step.data[other_idx].real)]
prob_w = [float(sv_step.probabilities()[target_idx] * 100)]

trace_rows = [
    f"| **Init** | - | - | `{amp_w[0]:+.4f}` | `{amp_other[0]:+.4f}` | `{prob_w[0]:.2f}%` |"
]

for k in range(1, k_opt + 1):
    # (1) Oracle 반단계: 부호만 -1 반전 (확률 불변)
    prev_pw = prob_w[-1]
    sv_step = sv_step.evolve(qc_ora)
    sign_pre = (-1) ** (k - 1)
    aw_o = float(sv_step.data[target_idx].real) * sign_pre
    ao_o = float(sv_step.data[other_idx].real) * sign_pre
    pw_o = float(sv_step.probabilities()[target_idx] * 100)

    labels.append(f"O{k}")
    is_oracle.append(True)
    amp_w.append(aw_o)
    amp_other.append(ao_o)
    prob_w.append(pw_o)

    # (2) Diffusion 반단계: 평균 반전으로 정답 진폭·확률 증폭
    sv_step = sv_step.evolve(qc_diff)
    sign_post = (-1) ** k
    aw_d = float(sv_step.data[target_idx].real) * sign_post
    ao_d = float(sv_step.data[other_idx].real) * sign_post
    pw_d = float(sv_step.probabilities()[target_idx] * 100)

    labels.append(f"D{k}")
    is_oracle.append(False)
    amp_w.append(aw_d)
    amp_other.append(ao_d)
    prob_w.append(pw_d)

    # [자가 검증] Oracle은 확률 불변, Diffusion은 k=1..12 동안 확률 단조 증가
    assert np.isclose(pw_o, prev_pw, atol=1e-12), f"O{k} 직후 확률이 변하지 않아야 함"
    assert pw_d > pw_o, f"D{k} 직후 정답 확률이 증가해야 함"

    trace_rows.append(
        f"| **$k={k}$** | `{aw_o:+.4f}` | `{pw_o:.2f}%` | `{aw_d:+.4f}` | `{ao_d:+.4f}` | **`{pw_d:.2f}%`** |"
    )

assert prob_w[-1] > 99.99, "12회 완료 후 정답 확률은 99.99% 이상이어야 함"

# [출력·시각화 전용 (양자 연산과 무관)] 13행 추적 표 및 2단 톱니·계단 그래프
display(Markdown(
    "**[관찰 1-2 결과] Toy $n=8$ 12회 반복(`Oracle -> Diffusion`) 진폭·확률 변화**\n\n"
    "| 단계 | Oracle 직후 $a_\\omega$ | Oracle 직후 $P(\\omega)$ | Diffusion 직후 $a_\\omega$ | 오답 진폭 $a_{\\text{other}}$ | Diffusion 직후 $P(\\omega)$ |\n"
    "| :---: | :---: | :---: | :---: | :---: | :---: |\n"
    + "\n".join(trace_rows)
))

x_idx = np.arange(len(labels))
ora_mask = np.array(is_oracle)
diff_mask = ~ora_mask

fig, axes = plt.subplots(2, 1, figsize=(12.6, 6.4), sharex=True)
axes[0].axhline(0.0, color="#868e96", linestyle=":", linewidth=1.0)
axes[0].plot(x_idx, amp_w, color="#0072ce", linewidth=1.5, zorder=2, label=r"Target amplitude $a_\omega$")
axes[0].scatter(x_idx[diff_mask], np.array(amp_w)[diff_mask], color="#0072ce", s=38, zorder=3, label="Init / After Diffusion (D)")
axes[0].scatter(x_idx[ora_mask], np.array(amp_w)[ora_mask], color="#e03131", s=38, zorder=3, label="After Oracle (O, sign flip)")
axes[0].plot(x_idx, amp_other, "--", color="#6c757d", linewidth=1.3, label=r"Non-target amplitude $a_{\mathrm{other}}$")
axes[0].set_ylabel("Amplitude")
axes[0].set_ylim(-1.15, 1.18)
axes[0].set_title(r"Step-by-Step Toy Grover ($n=8, k^*=12$): Oracle (Sign Flip) vs Diffusion (Amplitude Boost)")
axes[0].legend(loc="upper left", ncol=2, fontsize=8.5)
axes[0].grid(alpha=0.25)

axes[1].plot(x_idx, prob_w, color="#0072ce", linewidth=1.8, zorder=2)
axes[1].scatter(x_idx[diff_mask], np.array(prob_w)[diff_mask], color="#0072ce", s=42, zorder=3, label="Init / After D (Boost)")
axes[1].scatter(x_idx[ora_mask], np.array(prob_w)[ora_mask], color="#e03131", s=42, zorder=3, label="After O (Flat)")
for xi, lbl, pv in zip(x_idx, labels, prob_w):
    if lbl == "Init" or lbl.startswith("D"):
        axes[1].text(xi, pv + 3.2, f"{pv:.1f}%", ha="center", va="bottom", fontsize=7.8, color="#1a365d", fontweight="bold")
axes[1].set_ylabel("Target Probability (%)")
axes[1].set_xlabel("Half-Step Progression (Init -> O1 -> D1 -> ... -> O12 -> D12)")
axes[1].set_xticks(x_idx)
axes[1].set_xticklabels(labels, rotation=45, ha="right", fontsize=8.5)
axes[1].set_ylim(-6, 115)
axes[1].legend(loc="upper left", fontsize=8.5)
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## [관찰 2] 반복 횟수 $k$에 따른 증폭과 과회전(Over-rotation) 곡선

- **이론 공식**: 1회당 $+2\theta$ 회전하므로 $k$회 반복 후 정답 확률은 $P_{\text{win}}(k) = \sin^2\big((2k+1)\theta\big)$ ($\theta = \arcsin\frac{1}{\sqrt{256}}$).
- **핵심 Qiskit 명령**: `sv = sv.evolve(qc_step)`
  - $k=0\sim 30$ 회로를 매번 새로 빌드하지 않고, 현재 상태벡터 `sv`에 1회 반복 회로 `qc_step`($G = DO_\omega$)을 한 단계씩 추가 적용.
- **출력 구성**:
  1. 핵심 지점 요약 표: $k=0$(초기 `0.39%`), $k^*=12$(정점 `99.995%`), $k=25$(과회전 바닥 `약 0.23%`) 비교.
  2. 과회전 곡선 그래프 (가로축 $k=0\sim 30$, 세로축 확률 %): 회색 점선(이론 공식)과 파란 점(Qiskit `sv.evolve` 결과)의 일치 여부 및 $k=12$ 초과 시 확률 하락 구간 확인.

In [ ]:
# [Qiskit 1회 반복 회로 구성] 초기 상태 |s> 회로와 G = D O_omega 1단계 회로 분리
qc_init_8 = QuantumCircuit(n)
qc_init_8.h(range(n))

qc_toy_step = QuantumCircuit(n)
apply_toy_oracle(qc_toy_step, target)
apply_diffusion(qc_toy_step, list(range(n)))

# [Qiskit 상태벡터 순차 전진 (sv.evolve)] k=0..30 정답 확률 기록
ks = list(range(0, 31))
sim_probs = []
sv_curr = Statevector.from_instruction(qc_init_8)
for k in ks:
    sim_probs.append(float(sv_curr.probabilities()[target_idx]))
    sv_curr = sv_curr.evolve(qc_toy_step)

# [이론 곡선 계산 및 자가 검증 (NumPy)] 시뮬레이션과 이론 sin^2((2k+1)*theta) 대조
theta = np.arcsin(1.0 / np.sqrt(N))
k_dense = np.linspace(0, 30, 300)
theory_curve = np.sin((2 * k_dense + 1) * theta) ** 2
assert np.allclose(sim_probs, np.sin((2 * np.array(ks) + 1) * theta) ** 2, atol=1e-10)

# [출력·시각화 전용 (양자 연산과 무관)] 핵심 지점 표 및 이론 vs 시뮬레이션 곡선 그래프
display(Markdown(
    f"**[관찰 2 결과] Toy $n=8$ 반복 횟수별 핵심 지점**\n\n"
    f"| 반복 횟수 ($k$) | 정답 확률 | 의미 |\n"
    f"| :---: | :---: | :--- |\n"
    f"| $k=0$ | `{sim_probs[0]*100:.2f}%` | 초기 균등 중첩 ($1/256$) |\n"
    f"| **$k=12$ ($k^*$)** | **`{sim_probs[12]*100:.3f}%`** | 정답축 $\\ket{{\\omega}}$ ($90^\\circ$)에 가장 가까운 정점 |\n"
    f"| **$k=25$** | **`{sim_probs[25]*100:.3f}%`** | $180^\\circ$ 부근까지 과회전하여 처음보다도 낮아짐 |"
))

fig, ax = plt.subplots(figsize=(10.5, 4.0))
ax.plot(k_dense, theory_curve * 100, "--", color="#6c757d", label=r"Theory $\sin^2((2k+1)\theta)$")
ax.plot(ks, np.array(sim_probs) * 100, "o", color="#0072ce", markersize=5.5, label="Qiskit Simulation (sv.evolve)")
ax.axvline(12, color="#00a896", linestyle=":", linewidth=1.6, label=rf"Optimal $k^*=12$ ({sim_probs[12]*100:.3f}%)")
ax.axvline(25, color="#c92a2a", linestyle=":", linewidth=1.6, label=rf"Over-rotation $k=25$ ({sim_probs[25]*100:.2f}%)")
ax.set_title(r"Grover Amplitude Amplification & Over-rotation ($n=8, N=256, M=1$)")
ax.set_xlabel("Number of Grover Iterations ($k$)")
ax.set_ylabel("Target Probability (%)")
ax.set_ylim(-4, 108)
ax.legend(loc="upper right")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## [관찰 3-1] 검산 규칙 기반 Grover: 8-노드 2-색칠 오라클 회로 읽기 (16큐비트)

- **문제 정의**: 일렬로 연결된 8개 노드($q_0 - q_1 - \cdots - q_7$)에 두 색(`0`/`1`)을 배정할 때 인접 노드의 색이 모두 달라야 함. 정답은 `01010101`, `10101010` 2개($M=2$)이나, 회로에는 정답 대신 검산 규칙만 구현.
- **16큐비트 역할 분담**:
  1. **메인 큐비트 `q_0 ~ q_7` (8개)**: 후보 색칠 $x$ 저장.
  2. **임시 검산 보조 큐비트 `q_8 ~ q_14` (7개)**: 인접 7쌍의 색이 다르면 `1`, 같으면 `0` 기록(`qc.cx(e, 8+e)`, `qc.cx(e+1, 8+e)`).
  3. **부호 반전 도우미 `q_15` (1개)**: 초기 $X, H$로 $\ket{-}$ 상태 준비. 7개 보조 큐비트가 모두 `1`일 때만 `qc.mcx([8..14], 15)`가 발동해 $X\ket{-} = -\ket{-}$ 성질로 메인 상태 전체 부호만 $-1$ 반전.
- **출력 구성**: `초기화(1회)` 뒤 이어지는 **반복 구간(`Repeat x8`, 파란 점선 박스)** 안에 오라클과 Diffusion을 함께 표시한 16큐비트 전체 회로도 출력.
  1. 초기화 (1회): `q_0..q_7`에 `H` 적용($\ket{s}$), `q_15`에 `X, H` 적용($\ket{-}$).
  2. `1. Compute` (오라클): 계단식 `CX`로 인접 색 비교 결과를 `q_8..q_14`에 기록.
  3. `2. Flip` (오라클): `q_8..q_14`가 `q_15`를 제어해 정답 후보 부호 반전.
  4. `3. Uncompute` (오라클): `1. Compute`의 역순 `CX`로 `q_8..q_14`를 `0`으로 복원.
  5. `4. Diffusion D`: 메인 `q_0..q_7`에만 $H-X-\mathrm{MCZ}-X-H$ 적용(위 `Repeat x8` 구간을 총 8회 반복).

In [ ]:
# [Qiskit 검산 오라클 함수 정의] Compute -> Flip -> Uncompute
def apply_2coloring_oracle(
    qc: QuantumCircuit, uncompute: bool = True, with_barriers: bool = False
) -> None:
    '''8-노드 2-색칠 검산 오라클 (메인 0..7, 임시 검산 8..14, 부호 반전 도우미 15)'''
    # 1. 검산 (Compute): 7개 간선의 색이 다르면 보조 큐비트 8+e가 1이 됨
    for e in range(7):
        qc.cx(e, 8 + e)
        qc.cx(e + 1, 8 + e)
    if with_barriers:
        qc.barrier(label="2. Flip")

    # 2. 부호 반전 (Phase Flip): 7개 간선이 모두 1일 때만 |-> 상태의 q_15에 MCX 적용
    qc.mcx(list(range(8, 15)), 15)
    if with_barriers:
        qc.barrier(label="3. Uncompute")

    # 3. 임시 기록 정리 (Uncompute): 1단계 CX를 역순으로 재적용해 q_8..q_14를 |0>으로 복원
    if uncompute:
        for e in reversed(range(7)):
            qc.cx(e + 1, 8 + e)
            qc.cx(e, 8 + e)


# [Qiskit 회로도 구성: 초기화(1회) + 반복 1회(Oracle + Diffusion D)]
qc_preview = QuantumCircuit(16)
qc_preview.h(range(8))          # 메인 8큐비트 균등 중첩 |s>
qc_preview.x(15)
qc_preview.h(15)                # 부호 반전 도우미 q_15를 |-> 상태로 준비
qc_preview.barrier(label="Repeat x8 START")
apply_2coloring_oracle(qc_preview, uncompute=True, with_barriers=True)
qc_preview.barrier(label="4. Diffusion D")
apply_diffusion(qc_preview, list(range(8)))
qc_preview.barrier(label="Repeat x8 END")

fig_circ = qc_preview.draw("mpl", fold=-1, scale=0.52)

# [출력·시각화 전용 (양자 연산과 무관)] 반복 구간(Repeat x8) 파란 점선 박스 및 제목 표시
from matplotlib.patches import Rectangle

ax_circ = fig_circ.axes[0]
label_pos = {t.get_text(): t.get_position()[0] for t in ax_circ.texts}
x_start = label_pos.get("Repeat x8 START", 2.8) - 0.35
x_end = label_pos.get("Repeat x8 END", ax_circ.get_xlim()[1] - 1.0) + 0.35
y_min, y_max = ax_circ.get_ylim()
ax_circ.add_patch(
    Rectangle(
        (x_start, y_min + 0.15),
        x_end - x_start,
        (y_max - y_min) - 0.15,
        fill=True,
        facecolor="#ebf8ff",
        alpha=0.22,
        edgecolor="#0072ce",
        linestyle="--",
        linewidth=2.0,
        zorder=0,
    )
)
ax_circ.set_ylim(y_min - 0.15, y_max + 0.95)
ax_circ.text(
    0.5 * (x_start + x_end),
    y_max + 0.42,
    "<<<  Repeat x8 (k* = 8): Oracle (Compute -> Flip -> Uncompute) + Diffusion D  >>>",
    ha="center",
    va="center",
    fontsize=10.5,
    fontweight="bold",
    color="#0055a5",
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "#ebf8ff", "edgecolor": "#0072ce", "lw": 1.2},
)
fig_circ.suptitle("16-Qubit 8-Node 2-Coloring Grover: Init (1x) + [Oracle + Diffusion D] (x8)", fontsize=12)
display(fig_circ)

## [관찰 3-2] 16큐비트 실행($k^*=8$) 및 임시 보조 큐비트 정리 비교 실험

- **실험 조건**: $N=256, M=2$이므로 $k^* = \left\lfloor \frac{\pi}{4}\sqrt{\frac{256}{2}} \right\rfloor = 8$회 실행.
- **Uncomputation(임시 기록 정리) 생략 시 간섭 붕괴 원리** (2일차 얽힘 개념 연결):
  - `uncompute=True`: 모든 후보의 `q_8..q_14`가 `0000000`으로 복원되어 256개 후보가 동일 배경 위에서 만나 정상 간섭 발생.
  - `uncompute=False`: 후보마다 `q_8..q_14` 값이 달라 메인과 보조 큐비트가 얽힌 채 남음. 배경이 다른 후보끼리는 서로 다른 직교 공간에 격리되어 Diffusion을 적용해도 진폭이 제대로 섞이지 않음.
- **핵심 Qiskit 명령**: `sv.probabilities(qargs=list(range(8)))`
  - 16큐비트 상태벡터($65,536$개 항)에서 보조 큐비트 확률을 합산하고 메인 큐비트 `0..7`의 256개 확률만 추출.
- **출력 구성**:
  1. 비교 표: 정상(`99.56%`) vs 정리 생략 비교. 본 회로는 같은 보조 큐비트를 재사용하므로 짝수 회차에 일부 XOR 기록이 상쇄되어 0%까지 내려가지는 않으나, 홀수 회차 간섭 저하로 성공 확률이 약 `50.9%`로 절반 가까이 하락(반복마다 새 보조 큐비트를 썼다면 증폭 불가).
  2. 좌/우 막대 그래프 (인덱스 `0~255`): 왼쪽(`49.8%`씩 두 기둥) 대비 오른쪽은 두 기둥이 `25.5%` 수준으로 낮아지고 오답 구간으로 진폭이 분산됨.

In [ ]:
# [Qiskit 16큐비트 2-색칠 회로 빌더] 초기 상태 회로와 1회 반복 회로 분리
def build_2coloring_init_and_step(uncompute: bool = True) -> tuple[QuantumCircuit, QuantumCircuit]:
    qc_init = QuantumCircuit(16)
    qc_init.h(range(8))     # 메인 8큐비트 균등 중첩
    qc_init.x(15)
    qc_init.h(15)           # 도우미 q_15를 |-> 상태로 준비

    qc_step = QuantumCircuit(16)
    apply_2coloring_oracle(qc_step, uncompute=uncompute)
    apply_diffusion(qc_step, list(range(8)))
    return qc_init, qc_step


def run_2coloring(iterations: int, uncompute: bool = True) -> Statevector:
    qc_init, qc_step = build_2coloring_init_and_step(uncompute=uncompute)
    sv = Statevector.from_instruction(qc_init)
    for _ in range(iterations):
        sv = sv.evolve(qc_step)
    return sv


# [Qiskit 상태벡터 실행] 정리 포함(uncompute=True) vs 정리 생략(uncompute=False)
t0 = time.perf_counter()
sv_kb = run_2coloring(iterations=8, uncompute=True)
p_kb = sv_kb.probabilities(qargs=list(range(8)))
elapsed_kb = time.perf_counter() - t0

sv_no_uncomp = run_2coloring(iterations=8, uncompute=False)
p_no_uncomp = sv_no_uncomp.probabilities(qargs=list(range(8)))

# [자가 검증] 두 정답(01010101, 10101010) 확률 합 확인
idx_a, idx_b = int("01010101", 2), int("10101010", 2)
sum_ok = float(p_kb[idx_a] + p_kb[idx_b])
sum_bad = float(p_no_uncomp[idx_a] + p_no_uncomp[idx_b])
assert sum_ok > 0.99, "Uncomputation 포함 시 두 정답 확률 합은 99% 이상이어야 함"
assert sum_ok - sum_bad > 0.40, "Uncomputation 생략 시 확률 합이 크게 하락해야 함"

# [출력·시각화 전용 (양자 연산과 무관)] 실험 조건 비교 표 및 좌/우 분포 막대 그래프
display(Markdown(
    f"**[관찰 3-2 결과] 16큐비트 8-노드 2-색칠 ($k^*=8$회, 실행 시간 `{elapsed_kb:.2f}s`)**\n\n"
    f"| 실험 조건 | $P(\\texttt{{01010101}})$ (idx={idx_a}) | $P(\\texttt{{10101010}})$ (idx={idx_b}) | 두 정답 확률 합 | 해석 |\n"
    f"| :--- | :---: | :---: | :---: | :--- |\n"
    f"| **정상 (임시 기록 정리 O)** | `{p_kb[idx_a]*100:.2f}%` | `{p_kb[idx_b]*100:.2f}%` | **`{sum_ok*100:.2f}%`** | 정답 2개 자동 검출 성공 |\n"
    f"| **정리 생략 (Uncompute X)** | `{p_no_uncomp[idx_a]*100:.2f}%` | `{p_no_uncomp[idx_b]*100:.2f}%` | `{sum_bad*100:.2f}%` | 보조 큐비트 얽힘으로 증폭 절반 붕괴 |"
))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), sharey=True)
axes[0].bar(range(256), p_kb * 100, width=1.2, color="#00a896")
axes[0].set_title(rf"Normal (uncompute=True, $k^*=8$): Sum {sum_ok*100:.2f}%")
axes[0].set_xlabel("Main 8-Qubit State Index (0 ~ 255)")
axes[0].set_ylabel("Probability (%)")
axes[0].set_ylim(0, 60)

axes[1].bar(range(256), p_no_uncomp * 100, width=1.2, color="#c92a2a")
axes[1].set_title(rf"Without Cleanup (uncompute=False, $k=8$): Sum {sum_bad*100:.2f}%")
axes[1].set_xlabel("Main 8-Qubit State Index (0 ~ 255)")
plt.tight_layout()
plt.show()

## [관찰 3-3] 다중 해($M=2$) 반복 횟수별 증폭 곡선 ($k=0\sim 18$)

- **다중 해 회전 특성**: $M=2$에서는 $\sin\theta = \sqrt{2/256}$으로 1회당 회전 각도 $2\theta$가 커져 최적 반복 횟수가 $k^* = \left\lfloor \frac{\pi}{4}\sqrt{N/M} \right\rfloor = 8$회로 단축됨.
- **실습 내용**: 16큐비트 회로를 `sv.evolve()`로 $k=0\sim 18$ 전진시키며 두 정답(`01010101` + `10101010`) 확률 합 추적.
- **출력 구성**: 회색 점선(이론 곡선 $\sin^2((2k+1)\theta)$)과 청록색 점(Qiskit 16큐비트 시뮬레이션)이 일치하며 $k^*=8$회에서 `99.56%` 정점 도달 후 과회전으로 하강하는 그래프 출력.

In [ ]:
# [Qiskit 상태벡터 순차 전진 (16큐비트, k=0..18)]
qc_init_2c, qc_step_2c = build_2coloring_init_and_step(uncompute=True)
ks_kb = list(range(0, 19))
kb_curve = []

sv_tmp = Statevector.from_instruction(qc_init_2c)
for k in ks_kb:
    pk = sv_tmp.probabilities(qargs=list(range(8)))
    kb_curve.append(float(pk[idx_a] + pk[idx_b]) * 100)
    sv_tmp = sv_tmp.evolve(qc_step_2c)

# [이론 곡선 계산 및 자가 검증 (NumPy)]
theta_kb = np.arcsin(np.sqrt(2.0 / 256.0))
theory_kb_dots = np.sin((2 * np.array(ks_kb) + 1) * theta_kb) ** 2 * 100
assert np.allclose(kb_curve, theory_kb_dots, atol=1e-10)

# [출력·시각화 전용 (양자 연산과 무관)] 다중 해 이론 vs 시뮬레이션 곡선 그래프
k_grid = np.linspace(0, 18, 200)
fig, ax = plt.subplots(figsize=(10.5, 4.0))
ax.plot(k_grid, np.sin((2 * k_grid + 1) * theta_kb) ** 2 * 100, "--", color="#6c757d", label="Theory (N=256, M=2)")
ax.plot(ks_kb, kb_curve, "o", color="#00a896", markersize=5.8, label="Qiskit 16-Qubit Sim (sv.evolve)")
ax.axvline(8, color="#0072ce", linestyle=":", linewidth=1.6, label=rf"Optimal $k^*=8$ ({kb_curve[8]:.2f}%)")
ax.set_title(r"Multi-Solution ($M=2$) Amplitude Amplification vs Iterations ($N=256$)")
ax.set_xlabel("Iterations ($k$)")
ax.set_ylabel("Combined Probability of 2 Solutions (%)")
ax.set_ylim(-4, 108)
ax.legend(loc="lower left")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## [관찰 4] 유한 측정(Shots)과 통계적 수렴 (Shot Noise)

- **배경**: 실제 양자 칩은 이론 확률 분포(`Statevector`)를 직접 읽지 못하고 회로를 1회 실행할 때마다 비트열 1개만 관측하므로, $S$ Shots 반복 측정으로 빈도 비율을 추정.
- **핵심 Qiskit 명령**: `sv_kb.sample_counts(S, qargs=list(range(8)))`
  - 16큐비트 상태벡터 `sv_kb`에서 보조 큐비트(`8..15`)를 제외한 메인 8큐비트만 `S`회 측정해 관측 횟수 딕셔너리 반환.
- **출력 구성**:
  1. Shot 수별 요약 표 (`64`, `512`, `2,048` Shots): 두 정답(`01010101`, `10101010`, 각 이론 49.8%) 및 기타 오답 관측 횟수·비율 비교.
  2. 3분할 막대 그래프: 붉은 점선(이론값 49.8%) 대비 `64 Shots`의 큰 표본 오차(Shot Noise)가 `2,048 Shots`로 갈수록 49.8% 선에 수렴하는 과정 확인.

In [ ]:
# [Qiskit 유한 측정 샘플링 (sample_counts)] 및 비율 집계 데이터 준비
shot_list = [64, 512, 2048]
sv_kb.seed(7)

shot_results = []
for s in shot_list:
    # (1) Qiskit 양자 측정 API 호출
    counts = sv_kb.sample_counts(s, qargs=list(range(8)))
    assert sum(counts.values()) == s

    # (2) 표·그래프용 비율 계산 (양자 연산과 무관)
    c_a = int(counts.get("01010101", 0))
    c_b = int(counts.get("10101010", 0))
    c_other = s - c_a - c_b
    ratios = [c_a / s * 100, c_b / s * 100, c_other / s * 100]
    shot_results.append((s, c_a, c_b, c_other, ratios))

# [출력·시각화 전용 (양자 연산과 무관)] Shot 수별 결과 표 및 3분할 막대 그래프
summary_rows = [
    f"| `{s}` | {c_a}회 ({ratios[0]:.1f}%) | {c_b}회 ({ratios[1]:.1f}%) | {c_other}회 ({ratios[2]:.1f}%) |"
    for s, c_a, c_b, c_other, ratios in shot_results
]
display(Markdown(
    "**[관찰 4 결과] Qiskit `sample_counts` 측정 횟수별 수렴 ($\\mathcal{O}(1/\\sqrt{S})$)**\n\n"
    "| Shots ($S$) | `01010101` (이론 49.8%) | `10101010` (이론 49.8%) | 기타 오답 (254개 합) |\n"
    "| :---: | :---: | :---: | :---: |\n"
    + "\n".join(summary_rows)
))

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), sharey=True)
for ax, (s, _, _, _, ratios) in zip(axes, shot_results):
    ax.bar(["01010101", "10101010", "Others"], ratios, color=["#0072ce", "#00a896", "#adb5bd"])
    ax.axhline(49.78, color="#c92a2a", linestyle="--", linewidth=1.2, label="Theory (49.8%)")
    ax.set_title(f"{s} Shots")
    ax.set_ylim(0, 72)
    if s == 64:
        ax.set_ylabel("Observed Ratio (%)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3일차 핵심 정리 & Qiskit 복붙 체크리스트

### 1. 오늘 확인한 4가지 원리
1. **진폭 증폭과 과회전**: $k^* = \left\lfloor\frac{\pi}{4}\sqrt{\frac{N}{M}}\right\rfloor$에서 정답 확률이 정점(`99.995%` / `99.56%`)에 도달하며, 초과 반복 시 확률이 다시 감소.
2. **검산 규칙 기반 확장**: 정답을 하드코딩하지 않고 $f(x)=1$ 검산 회로만으로 다중 해($M=2$)까지 동시 탐색 가능.
3. **임시 보조 큐비트 정리 (Uncomputation)**: 임시 큐비트를 $\ket{0}$으로 복원하지 않으면 메인 큐비트와의 얽힘 때문에 다음 Diffusion 간섭이 붕괴.
4. **Shot Noise 수렴**: 유한 횟수 $S$ 측정의 표본 오차는 측정 횟수를 늘릴수록 이론 분포에 수렴.

### 2. Qiskit 최소 패턴 (AI 코드 검산·복붙용)
```python
qc = QuantumCircuit(n)                  # 1. n큐비트 회로 생성
qc.h(range(n)); qc.cx(0, 1)             # 2. 게이트 배치 (또는 ZGate().control(k))
qc.draw("mpl")                          # 3. 회로도 확인 (Little-Endian 주의)
sv = Statevector.from_instruction(qc)   # 4. 상태벡터 계산 (연속 전진은 sv.evolve)
counts = sv.sample_counts(1000)         # 5. 유한 Shots 측정 분포 추출
```

- **4일차 예고**: 파이썬 코드 대신 웹 브라우저 GUI에서 회로를 조립·실행하고 `Compare` 기능으로 다중 실험 결과를 비교하는 QubitSquare 플랫폼 종합 실습 진행.